## 🔄 Incremental Ingestion: Setting Up Azure SQL Database

### What is Incremental Ingestion?
Incremental ingestion (or delta loading) is the process of loading only new or updated data since the last pipeline run, rather than extracting the entire dataset every time. 

*Note on Modern Architecture*: While the traditional approach heavily relied on **watermark columns** (like `LastModifiedDate`) stored in control tables, modern data engineering is shifting towards **Lakehouse architectures** (like Delta Lake on Databricks or Microsoft Fabric), which handle incremental changes (ACID transactions, time travel, and UPSERTs) much more efficiently under the hood. However, for our source system, we still need to set up an Azure SQL Database.

### ➕ Steps to Create an Azure SQL Database

To practice ingestion from a relational database, we first need to provision an Azure SQL Database.

**1. Create a New Resource**
* Log in to the Azure Portal.
* Search for **Azure SQL** in the top search bar and click on it.
* Click **+ Create** or **+ New**.
* You will see three options: *SQL databases*, *SQL managed instances*, and *SQL virtual machines*. 
  * **SQL databases (Our Choice)**: Fully managed PaaS. Lowest cost and zero maintenance. Perfect for simple data projects.
  * **SQL managed instances**: For "lift-and-shift" enterprise migrations needing legacy SQL features. (Too expensive/complex for us).
  * **SQL virtual machines**: IaaS (a full server). Gives OS-level control but requires high maintenance. (Overkill for us).
* Under **SQL databases**, click **Create**.

**2. Configure the Server & Basics**
* **Subscription & Resource Group**: Select your project's resource group.
* **Database name**: `adfprojectdb`
* **Server**: Click **Create new** to set up the logical server hosting the database.
  * **Server name**: `adfproject` (Note: Must be globally unique, so you may need to add numbers).
  * **Location**: Choose the same region as your Data Factory.
  * **Authentication method**: Select **Use both SQL and Microsoft Entra authentication**.
  * **Set server admin login**: `adminadmin`
  * **Password**: `12345` (and confirm password `12345` — *Note: In a real project, use a strong password!*)
  * Click **OK** to save the server setup.
* **Workload environment**: Select **Development** (to keep costs low, rather than Production. Development optimizes for cost-savings in Proof-of-Concepts, whereas Production defaults to expensive high-availability setups).

**3. Configure Compute and Storage**
* Under the **Compute + storage** section, click **Configure database**.
* **Service tier**: Choose **Serverless** (Serverless auto-pauses and stops billing you when inactive. Provisioned charges a fixed hourly rate even when idle).
* **Compute**: Scale down to **1 vCore**.
* **Data max size**: Scale down to **1 GB**.
* Click **Apply**.

**4. Configure Networking & Connectivity**
* Move to the **Networking** tab.
* **Connectivity method**: Select **Public endpoint**.
* **Firewall rules**:
  * Allow Azure services and resources to access this server: **Yes** (Crucial so ADF can talk to it).
  * Add current client IP address: **Yes** (So you can log in from your local machine).

**5. Additional Settings & Review**
* **Security**: For Microsoft Defender for SQL, select **Not now**.
* Click on **Review + create** at the bottom.
* Validate your configuration and click **Create** to deploy your Azure SQL Database! 

✅ **Result**: Once deployed, you will have a fully functional relational database ready to host the tables we will incrementally ingest into our Data Lake.

## 🔄 Incremental Ingestion: Source Table and Watermark Setup

### ➕ Step 1: Create the Source Table in Azure SQL Database
First, we need to create our source table and populate it with some dummy data to simulate a real-world scenario.

1. Go to your Azure SQL Database (`adfprojectdb`) in the Azure Portal.
2. Click on **Query editor (preview)** on the left menu.
3. Log in using your SQL authentication (`adminadmin` / `12345`).
4. **Query 1**: Paste and execute the following to create the table:
   ```sql
   CREATE TABLE FactBookings (
       booking_id INT PRIMARY KEY,
       passenger_id INT,
       flight_id INT,
       airline_id INT,
       origin_airport_id INT,
       destination_airport_id INT,
       booking_date DATE,
       ticket_cost DECIMAL(10,2),
       flight_duration_mins INT,
       checkin_status VARCHAR(10)
   );
5. Run Query 2 (Insert Data)
INSERT INTO FactBookings VALUES (1, 90, 187, 8, 6, 3, '2025-04-20', 167.19, 178, 'Yes');
INSERT INTO FactBookings VALUES (2, 49, 110, 8, 9, 8, '2025-01-30', 475.86, 256, 'Yes');
INSERT INTO FactBookings VALUES (3, 93, 187, 10, 3, 9, '2025-01-23', 399.53, 607, 'Yes');
INSERT INTO FactBookings VALUES (4, 18, 149, 3, 9, 2, '2025-03-23', 809.63, 306, 'No');
INSERT INTO FactBookings VALUES (5, 7, 190, 2, 8, 1, '2025-02-01', 933.3, 576, 'No');

### ➕ Step 2: Initialize Data Lake for Watermarking
We will use a file-based watermarking technique to track the **available max timestamp** and only load new data.

1. Go to your **Azure Data Lake Storage Gen2** account > `bronze` container.
2. Create a new directory named: `monitor`.
3. Inside `monitor`, create two sub-directories: `emptyjson` and `lastload`.
4. On your local machine, create an empty JSON file named `empty.json` (containing only `{}`). **Upload** it to the `emptyjson` directory.
5. On your local machine, create another JSON file named `last_load.json` (containing `{"last_load": "1900-01-01"}`). **Upload** it to the `lastload` directory.

### ⏱️ Checking the Time Format (Optional but Recommended)
Before creating your final watermark file, you can run a quick test to see the exact UTC timestamp format that Azure Data Factory generates.

1. Go to **Azure Data Factory Studio** and open your new pipeline (`SQLtoDatalake`).
2. Drag a **Set variable** activity onto the canvas.
3. Click on the blank canvas background, go to the **Variables** tab at the bottom, and click **+ New** to create a temporary variable:
   * **Name**: `v_now`
   * **Type**: `String`
4. Click on your **Set variable** activity on the canvas.
5. Go to the **Variables** tab of the activity, select `v_now` from the dropdown, and for the **Value**, click *Add dynamic content* and enter:
   `@utcnow()`
6. Click **Debug** and check the activity output. 
   * *Why?* This shows you the exact UTC timestamp string format ADF uses, which helps you understand how to correctly format your `last_load.json` dates.
7. You can safely **delete** this Set variable activity after viewing the output.


### ➕ Step 3: Configure Lookup Activities
Now we set up the pipeline to read our start date (watermark) and end date (latest source data).

1. **Lookup 1 (`last_load`)**:
   * Drag a **Lookup** activity onto the `SQLtoDatalake` canvas and name it `last_load`.
   * **Dataset**: Create or select a JSON dataset pointing exactly to the `monitor/lastload/last_load.json` file.
   * Make sure **First row only** is checked.
2. **Lookup 2 (`latest_load`)**:
   * Drag a second **Lookup** activity and name it `latest_load`.
   * **Dataset**: Connect to your Azure SQL Database (`ds_sql_source`).
   * **Query**: Choose the **Query** option instead of Table and enter:
     `SELECT MAX(booking_date) AS latest_load FROM dbo.FactBookings`

### ➕ Step 4: The Main Copy Activity (Extract Data)
We copy only the data that falls *between* our last load date and our latest load date.

1. Drag a **Copy data** activity onto the canvas and name it `copysqldata`. 
2. Connect the success paths (green boxes) from **both** Lookup activities to this Copy activity.
3. **Source Tab**:
   * Select your Azure SQL dataset (`ds_sql_source`).
   * Choose **Query** and inject the dynamic lookup values using this expression:
     ```sql
     SELECT * FROM dbo.FactBookings 
     WHERE booking_date > '@{activity('last_load').output.firstRow.last_load}' 
     AND booking_date <= '@{activity('latest_load').output.firstRow.latest_load}'
     ```
4. **Sink Tab**:
   * Select a new ADLS Gen2 **Parquet** dataset.
   * Point it to your bronze folder (e.g., `bronze/SQL/`). 
   * *Crucial*: **Do not specify a filename**. Leave it blank so ADF automatically generates a unique Parquet file for each incremental run.

### ➕ Step 5: Update the Watermark File
Finally, we must update the `last_load.json` file with the new maximum date so the next pipeline run knows where to start.

1. Drag another **Copy data** activity and name it `watermark`. Connect it to the success path (green box) of the `copysqldata` activity.
2. **Source Tab**:
   * Select a dataset pointing to your `monitor/emptyjson/empty.json` file.
   * Scroll down to **Additional columns** and click **+ New**.
   * **Name**: `last_load`
   * **Value**: Click *Add dynamic content* and paste: `@activity('latest_load').output.firstRow.latest_load`
3. **Sink Tab**:
   * Select the dataset pointing to your `monitor/lastload/last_load.json` file.
   * *How it works*: This step takes the empty JSON file, injects the new date into a column named `last_load`, and securely overwrites the old watermark file!

✅ **Execution & Testing**: 
Click **Debug** to run your pipeline. 
* **Run 1**: It will copy all rows (since the start date is 1900) and update the watermark file to the max date.
* **Run 2**: If you debug again immediately without adding new SQL data, it will dynamically recognize the updated watermark date and copy **0 rows**, proving your incremental logic works perfectly!